# Eco-Scale: Prediction of Safe Server Shutdowns\n## Time-Series & Cluster-Level Regression\n\nInstead of predicting binary load on a single server, this notebook predicts the **exact number of servers** out of a 1000-node cluster that can be safely powered down while keeping overall capacity at a safe 75% limit.

In [5]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

### 1. Data Loading

In [13]:
df = pd.read_csv('cluster_telemetry.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df.head()

,timestamp,hour_of_day,day_of_week,avg_cpu_load,avg_memory_load,network_traffic_gbps,safe_shutdown_count
0,2026-03-01 00:00:00,0,6,10.895113,10.000000,6.742934,849
1,2026-03-01 01:00:00,1,6,14.228484,10.000000,6.865948,801
2,2026-03-01 02:00:00,2,6,7.395206,10.968742,2.535847,902
3,2026-03-01 03:00:00,3,6,5.000000,10.000000,1.000000,923
4,2026-03-01 04:00:00,4,6,8.522652,10.000000,3.197691,872


### 2. Exploratory Data Analysis (EDA) & Feature Relationships

In [14]:
# Time-Series View of Cluster Load
fig = px.line(
    df[:168],
    x='timestamp',
    y=['avg_cpu_load', 'safe_shutdown_count'],
    title='1-Week Snapshot: Inverse Relationship between CPU Load & Servers to Shutdown'
)
fig.show()

# Correlation Heatmap
corr = df.drop('timestamp', axis=1).corr()
fig2 = px.imshow(
    corr,
    text_auto=True,
    aspect="auto",
    title="Feature Correlation Heatmap"
)
fig2.show()

### 3. Model Training (Random Forest Regressor)

In [15]:
features = ['hour_of_day', 'day_of_week', 'avg_cpu_load', 'avg_memory_load', 'network_traffic_gbps']
X = df[features]
y = df['safe_shutdown_count']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

### 4. Evaluation & Feature Importance

In [16]:
preds = model.predict(X_test)
print(f"Mean Absolute Error (MAE): {mean_absolute_error(y_test, preds):.2f} servers")
print(f"R^2 Score: {r2_score(y_test, preds):.4f}")

importances = pd.DataFrame({
    'Feature': features,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

fig3 = px.bar(importances, x='Importance', y='Feature', orientation='h', title='Feature Importance in Random Forest Model')
display(fig3)

Mean Absolute Error (MAE): 5.32 servers
R^2 Score: 0.9994


### 5. Model Saving

In [17]:
joblib.dump(model, 'rf_cluster_model_notebook.pkl')
print("Model saved successfully.")

Model saved successfully.
